In [ ]:
from pathlib import Path
import pandas as pd

# Repository root
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Input and output folder paths
input_note_folder = ROOT / "data" / "raw" / "BPSD_v2" / "2_Annotations" / "ann_score_note"
input_chord_folder = ROOT / "data" / "raw" / "BPSD_v2" / "2_Annotations" / "ann_score_chord"
output_note_folder = ROOT / "data" / "processed" / "cleaned_data" / "ann_score_note_clean"
output_chord_folder = ROOT / "data" / "processed" / "cleaned_data" / "ann_score_chord_clean"

# Ensure output folders exist
output_note_folder.mkdir(parents=True, exist_ok=True)
output_chord_folder.mkdir(parents=True, exist_ok=True)

def clean_and_save_all_files(input_folder: Path, output_folder: Path) -> None:
    """
    Read all BPSD annotation CSV files from input_folder using semicolon separators
    and save cleaned CSV versions to output_folder.
    """
    for input_file_path in sorted(input_folder.glob("*.csv")):
        output_file_path = output_folder / f"{input_file_path.stem}_clean.csv"
        try:
            df = pd.read_csv(input_file_path, sep=";")
            df.to_csv(output_file_path, index=False)
            print(f"Saved: {output_file_path.name}")
        except Exception as e:
            print(f"Error processing {input_file_path.name}: {e}")

clean_and_save_all_files(input_note_folder, output_note_folder)
clean_and_save_all_files(input_chord_folder, output_chord_folder)

In [ ]:
from pathlib import Path
import pandas as pd

# Repository root
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Define folder paths
note_folder = ROOT / "data" / "processed" / "cleaned_data" / "ann_score_note_clean"
chord_folder = ROOT / "data" / "processed" / "cleaned_data" / "ann_score_chord_clean"
output_folder = ROOT / "data" / "processed" / "cleaned_data" / "merged_data"

# Ensure the output folder exists
output_folder.mkdir(parents=True, exist_ok=True)

# Get list of files in both folders
note_files = {f.stem: f for f in note_folder.glob("*.csv")}
chord_files = {f.stem: f for f in chord_folder.glob("*.csv")}

# Process each matching pair of files
for file_name in sorted(note_files.keys() & chord_files.keys()):
    note_file_path = note_files[file_name]
    chord_file_path = chord_files[file_name]

    # Load the note and chord data
    note_data = pd.read_csv(note_file_path)
    chord_data = pd.read_csv(chord_file_path)

    # Ensure columns are numeric for proper comparison
    note_data["start_meas"] = pd.to_numeric(note_data["start_meas"])
    note_data["end_meas"] = pd.to_numeric(note_data["end_meas"])
    chord_data["start"] = pd.to_numeric(chord_data["start"])
    chord_data["end"] = pd.to_numeric(chord_data["end"])

    # Merge notes with chord annotations:
    # a chord is matched if its span fully contains the note span.
    merged_data = []

    for _, note_row in note_data.iterrows():
        note_start = note_row["start_meas"]
        note_end = note_row["end_meas"]

        matching_chords = chord_data[
            (chord_data["start"] <= note_start) &
            (chord_data["end"] >= note_end)
        ]

        if not matching_chords.empty:
            for _, chord_row in matching_chords.iterrows():
                merged_row = note_row.copy()
                for chord_col in chord_data.columns:
                    merged_row[chord_col] = chord_row[chord_col]
                merged_data.append(merged_row)
        else:
            merged_row = note_row.copy()
            for chord_col in chord_data.columns:
                merged_row[chord_col] = None
            merged_data.append(merged_row)

    merged_df = pd.DataFrame(merged_data)

    # Rename columns for clarity
    merged_df.rename(columns={
        "start_meas": "note_start",
        "end_meas": "note_end",
        "timeSig": "note_timeSig",
        "start": "chord_start",
        "end": "chord_end",
        "localkey": "chord_localkey"
    }, inplace=True)

    # Drop redundant column if applicable
    if "majmin" in merged_df.columns and "shorthand" in merged_df.columns:
        if merged_df["majmin"].equals(merged_df["shorthand"]):
            merged_df.drop(columns=["majmin"], inplace=True)

    output_file_path = output_folder / f"{file_name}_merged.csv"
    merged_df.to_csv(output_file_path, index=False)

    print(f"Processed and saved: {output_file_path.name}")